In [2]:
import os
import sys

assert "jax" not in sys.modules, "jax already imported: you must restart your runtime"
os.environ['XLA_FLAGS'] = "--xla_force_host_platform_device_count=4"

In [3]:
import jax
import jax.numpy as jnp
from jax.random import PRNGKey
from jax.typing import ArrayLike

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mtick
from matplotlib.ticker import FuncFormatter


import datetime as dt
import time
from epiweeks import Week

import arviz as az

import numpy as np
import numpyro
import numpyro.distributions as dist
from numpyro.infer.reparam import LocScaleReparam

from pyrenew import observation, model, process, latent, deterministic, randomvariable
from pyrenew.process import RandomWalk
from pyrenew.latent import (
    Infections,
    InfectionInitializationProcess,
    InitializeInfectionsZeroPad,
    InitializeInfectionsExponentialGrowth,
    InfectionsWithFeedback
)
from pyrenew.deterministic import DeterministicPMF, DeterministicVariable
from pyrenew.metaclass import RandomVariable
from pyrenew.randomvariable import DistributionalVariable, TransformedVariable, StaticDistributionalVariable
import pyrenew.transformation as t

/Users/jingjieong/Projects/PyRenew/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Util functions

In [4]:
def conv(a: ArrayLike, b: ArrayLike) -> ArrayLike:
    """
    Helper function for convolution.
    """
    
    m = len(a)
    n = len(b)
    c = nc = np.zeros(m + n - 1)

    for i in range(m):       # Python index starts at idx
        for j in range(n):
            c[i + j] += a[i] * b[j]  # no -1 needed
    return c

def gamma_convert_param(mean, sd):
    shape = (mean**2)/(sd**2)
    scale = (sd**2)/mean

    return shape, scale


## plot utils
def format_quarterly_axis(ax):
    """
    Format a Matplotlib time axis to show quarterly ticks (Jan, Apr, Jul, Oct),
    with the year label shown only under January.
    """
    locator = mdates.MonthLocator(bymonth=[1, 4, 7, 10])
    ax.xaxis.set_major_locator(locator)

    def _custom_fmt(x, pos):
        date = mdates.num2date(x)
        if date.month == 1:
            return f"{date.strftime('%b')}\n{date.year}"
        else:
            return date.strftime('%b')

    ax.xaxis.set_major_formatter(FuncFormatter(_custom_fmt))
    return ax

### Data

In [5]:
# covid_hosp_d = pd.read_excel("../sg_data/agg_covid_hosp_variant.xlsx", sheet_name="NEW_ACTIVE_HOSPITALISATION")
# covid_variant_d = pd.read_excel("../sg_data/agg_covid_hosp_variant.xlsx", sheet_name="SARS-COV-2_VARIANT_PROPORTION")

covid_ari_hosp_d = pd.read_excel("../sg_data/agg_ari_ili.xlsx")

# covid_hosp_d.rename(columns={"YEAR":"year","EPI-WEEK":"epi_week"}, inplace = True)
# covid_variant_d.rename(columns={"YEAR":"year","EPI-WEEK":"epi_week"}, inplace = True)

def process_weeks_d(df):
    # Apply get_dates once and combine
    date_temp = df.apply(get_dates, axis=1)
    df = pd.concat([df, date_temp], axis=1)

    # Convert to datetime
    df[['start_date', 'end_date']] = df[['start_date', 'end_date']].apply(pd.to_datetime)

    # Create year-week column
    df['year_week'] = df['year'].astype(str) + "-" + df['epi_week'].astype(str)

    return df

# Apply the Week conversion to each row
def get_dates(row):
    try:
        epi_week = Week(int(row['year']), int(row['epi_week']))
        return pd.Series({
            'start_date': epi_week.startdate(),
            'end_date': epi_week.enddate()
        })
    except Exception as e:
        return pd.Series({
            'start_date': None,
            'end_date': None
        })

# Apply the function and combine results
# covid_hosp_d = process_weeks_d(covid_hosp_d)
covid_ari_hosp_d = process_weeks_d(covid_ari_hosp_d)

# covid_hosp_d.columns = covid_hosp_d.columns.str.lower()

covid_ari_hosp_d['ari_covid_positivity'] = covid_ari_hosp_d['ari_covid_pos'] / covid_ari_hosp_d['ari_samples']

### Sample-size study: the simulated positivity files cover the first N_WEEKS rows (2023 wk1 .. 2025 wk53)
N_WEEKS = 157

def add_case_estimates(df):
    """Recompute the columns derived from ari_covid_positivity. Mutates and returns df."""
    #est covid case count at poly
    df['est_covid_cases'] = df['poly_ari_attendances'] * df['ari_covid_positivity']
    df['poly_ari_attendances_adjusted'] = df['poly_ari_attendances'] / df['working_days'] * 5.5
    df['est_covid_cases_adj'] = df['poly_ari_attendances_adjusted'] * df['ari_covid_positivity']
    return df

def apply_positivity(base_df, positivity_values, n_rows=N_WEEKS):
    """First n_rows of base_df with ari_covid_positivity replaced by positivity_values
    and the derived case columns recomputed. Returns a new frame (no chained assignment)."""
    pos = np.asarray(positivity_values, dtype=float)
    if pos.shape != (n_rows,):
        raise ValueError(f"expected {n_rows} positivity values, got {pos.shape}")
    d = base_df.iloc[:n_rows].copy()
    d['ari_covid_positivity'] = pos
    return add_case_estimates(d)

# Base frame: OBSERVED positivity, all rows. Swap in a simulated series with apply_positivity(covid_ari_hosp_d, values)
covid_ari_hosp_d = add_case_estimates(covid_ari_hosp_d)

# Merge hosp and ARI data
# covid_ari_hosp_d = covid_ari_d.merge(covid_hosp_d[['year_week','new_hospitalisation_cases']],how='left',on = ['year_week'])

# Variant attributed case count
# merged_df = pd.merge(covid_variant_d,covid_ari_d[['year','epi_week','est_covid_cases']], on = ['year','epi_week'],how = 'outer')
# merged_df['est_variant_cases'] = merged_df['LINEAGE_PROPORTION'] * merged_df['est_covid_cases']

# covid_variant_counts = merged_df.pivot_table(
#     index = ['year','epi_week'],
#     columns = 'LINEAGE_GROUP',
#     values = 'est_variant_cases',
#     fill_value = 0,
#     aggfunc='sum'
# ).reset_index()

# covid_variant_counts = process_weeks_d(covid_variant_counts)

### Set wave period
wave_period = {'start': ['2023-01-01','2023-07-01','2024-03-01','2024-09-01'],
               'end': ['2023-08-01','2024-04-01','2024-09-01','2025-10-01']}


### COVID Model 1: Jointly fit to hospitalisation and primary care cases
- setup delay distributions (generation time, infection to hospitalisation/primary care)
- setup ascertainment rate
- setup Rt (random walk), I0 seeding, infection process and latent process for hosp and primary care
- setup observation model

In [6]:
## Delays
# onset_to_hosp = pd.read_csv('../sg_data/dist/dist_onset_hosp.csv') # SG COVID 2020 Data
# onset_to_hosp_dist = onset_to_hosp.onset_hosp / sum(onset_to_hosp.onset_hosp)

onset_hosp_mean = 8
onset_hosp_sd = 2
onset_hosp_shape, onset_hosp_scale = gamma_convert_param(onset_hosp_mean, onset_hosp_sd) 

j = jnp.arange(0,14)
onset_hosp_dist = dist.Gamma(concentration=onset_hosp_shape, rate=1/onset_hosp_scale)
onset_hosp_raw = jnp.exp(onset_hosp_dist.log_prob(j))
onset_to_hosp_dist = onset_hosp_raw / onset_hosp_raw.sum()

## (2) COVID Generation time interval (mean and sd from Vattiatio et al DOI: 10.1016/j.epidem.2022.100657)
gen_time_mean = 3.3
gen_time_sd = 1.3
gen_time_shape, gen_time_scale = gamma_convert_param(gen_time_mean, gen_time_sd) 

x = jnp.arange(0, 101)
gamma_dist = dist.Gamma(concentration=gen_time_shape, rate=1/gen_time_scale)
gen_time_raw = jnp.exp(gamma_dist.log_prob(x))

gen_time_dist = gen_time_raw / gen_time_raw.sum() #normalise
gen_int_rv = DeterministicPMF(name="gen_int", value=gen_time_dist[:15])

## (3) infection to hospitalisation  - Convolute incubation with onset to hospitalisation
# source: Jantien et al (2022) DOI: 10.2807/1560-7917
x = jnp.arange(0, 11)  # 0:1:10
weibull_dist = dist.Weibull(concentration=1.5, scale=3.2)

incubation_raw = jnp.exp(weibull_dist.log_prob(x))
incub_norm = incubation_raw / incubation_raw.sum()

inf_hosp_dist = conv(incub_norm, onset_to_hosp_dist)

inf_hosp_int = DeterministicPMF(
    name="inf_hosp_int", value=inf_hosp_dist
)

## onset to primary care visits, https://www.thelancet.com/journals/lanwpc/article/PIIS2666-6065(23)00237-7/fulltext
x = jnp.arange(0,7)
ons_primary_dist = dist.Gamma(concentration = 1.2, rate = 1/0.9) #mean 1.3, sd = 1.2
ons_primary_raw = jnp.exp(ons_primary_dist.log_prob(x))
ons_primary_norm = ons_primary_raw / sum(ons_primary_raw)

# Infection to primary care visit convolution
inf_primary_dist = DeterministicPMF('inf_primary_dist', conv(incub_norm, ons_primary_norm))

# n_initialization_points = max(gen_int_rv.sample().size, inf_hosp_int_array.size, inf_shed_int_array.size) - 1
n_initialization_points = max(gen_int_rv.sample().size, inf_hosp_dist.size, inf_primary_dist().size) - 1

# Probability of symptomatic given COVID infections:
# https://www.thelancet.com/journals/laninf/article/PIIS1473-3099(20)30833-1/fulltext
# https://www.acpjournals.org/doi/10.7326/M20-3012?
p_sym = DistributionalVariable('p_sym',distribution = dist.Beta(40,60))

hosp_rate = DistributionalVariable(
    name='hosp_rate', distribution=dist.Uniform(1e-4, 0.1)
)

## Ascertainment rate for primary care visits
class Alpha_primary(RandomVariable):

    @staticmethod
    def validate():
        return None

    def sample(self, p_sym, n = None):

        # https://www.thelancet.com/journals/lanwpc/article/PIIS2666-6065%2823%2900237-7/fulltext?
        # ~75% seek primary care, later periods of COVID (2021-2022). Now COVID endemic, lesser testing and healthcare seeking.
        p_hseek_dist = DistributionalVariable('p_hseek', distribution = dist.Beta(60,40))
        p_hseek = p_hseek_dist.sample()

        p_visit_dist = DistributionalVariable('p_visit', distribution = dist.Beta(15,85))
        p_visit = p_visit_dist.sample()

        alpha_primary = numpyro.deterministic('alpha_primary',p_sym*p_hseek*p_visit)
        
        return alpha_primary

alpha_primary_rv = Alpha_primary()

# Latent severe cases (uses the hospitaladmission module)
latent_severeprimary = latent.SeverePrimary(
    infection_to_severe_interval_rv= inf_hosp_int,
    infection_to_primary_interval_rv=inf_primary_dist,
    infection_severe_ratio_rv= hosp_rate,
    infection_primary_ratio_rv=alpha_primary_rv,
    symptomatic_ratio_rv = p_sym
)


## (4) Set up Rt, I0, latent infection process and observation
# I0 = InfectionInitializationProcess(
#     "I0_initialization",
#     randomvariable.DistributionalVariable(
#         name="I0",
#         distribution=dist.LogNormal(0,1),
#     ),
#     InitializeInfectionsExponentialGrowth(
#         n_initialization_points,
#         deterministic.DeterministicVariable(name="rate", value=0.05),
#     ),
# )

# (2) Initial infections (inferred with a prior)
I0 = InfectionInitializationProcess(
    "I0_initialization",
    DistributionalVariable(name="I0", distribution=dist.LogNormal(2.5, 1)),
    InitializeInfectionsZeroPad(n_initialization_points),
)

# The random walk on log Rt, with an inferred s.d. Here, we
# construct a custom RandomVariable.
class RtRandomWalk(RandomVariable):
    def validate(self):
        pass

    def sample(self, n: int, **kwargs) -> tuple:
        sd_rt = numpyro.sample("Rt_random_walk_sd", dist.HalfNormal(0.025))

        # Random walk step
        step_rv = randomvariable.DistributionalVariable(
            name="rw_step_rv", distribution=dist.Normal(0, sd_rt)
        )

        rt_init_rv = randomvariable.DistributionalVariable(
            name="init_log_rt", distribution=dist.Normal(0, 0.2)
        )

        # Random walk process
        base_rv = process.RandomWalk(
            name="log_rt",
            step_rv=step_rv,
        )

        # Transforming the random walk to the Rt scale
        rt_rv = randomvariable.TransformedVariable(
            name="Rt_rv",
            base_rv=base_rv,
            transforms=t.ExpTransform(),
        )
        init_rt = rt_init_rv.sample()

        return rt_rv.sample(n=n, init_vals=init_rt, **kwargs)

rtproc = RtRandomWalk()

# Latent infection process (which will use 1 and 2)
# feedback_strength = DeterministicVariable(name="feedback_strength", value=0.05)
# feedback_strength = DeterministicVariable(name="feedback_strength", value=0.01)

inf_feedback_strength_rv = TransformedVariable(
    "inf_feedback",
    DistributionalVariable(
        "inf_feedback_raw",
        dist.LogNormal(jnp.log(0.1), 0.5),
    ),
    transforms=t.AffineTransform(loc=0, scale=1),
)

inf_feedback_pmf_rv = DeterministicVariable(name = 'inf_feedback_pmf', value = jnp.concatenate([jnp.array([1]),jnp.repeat(0,gen_int_rv().size - 1)]))

# latent_inf_feedback = InfectionsWithFeedback(
#     infection_feedback_strength=inf_feedback_strength_rv,
#     infection_feedback_pmf=inf_feedback_pmf_rv,
# )

latent_inf = latent.Infections()

#Observation model - hospitalisation
nb_conc_hosp_rv = TransformedVariable(
    "concentration_hosp",
    randomvariable.DistributionalVariable(
        name="concentration_raw_hosp",
        distribution=dist.TruncatedNormal(loc=0, scale=1, low=0.01),
    ),
    t.PowerTransform(-2),
)

obs_h = observation.NegativeBinomialObservation(
    "negbinom_hosp_rv",
    concentration_rv=nb_conc_hosp_rv,    
)

#Observation model - primary care cases
nb_conc_primary_rv = TransformedVariable(
    "concentration_primary",
    randomvariable.DistributionalVariable(
        name="concentration_raw_primary",
        distribution=dist.TruncatedNormal(loc=0, scale=1, low=0.01),
    ),
    t.PowerTransform(-2),
)

obs_p = observation.NegativeBinomialObservation(
    "negbinom_primary_rv",
    concentration_rv=nb_conc_primary_rv,    
)

## If using wastewater, specify distribution and variables

# onset_to_shed = pd.read_csv('../sg_data/dist/dist_ww_shedding.csv') # Shedding distribution

# onset_to_shed_dist = onset_to_shed / sum(onset_to_shed.Shedding)
# inf_shed_int_array = conv(incubation, onset_to_shed_dist.Shedding.values)[3:]
# inf_shed_int_array = inf_shed_int_array / sum(inf_shed_int_array)

# inf_shed_int = deterministic.DeterministicPMF(
#     name="inf_shed_int", value=inf_shed_int_array
# )

# #Time-varying alpha RV
# class TimeVaryingAlpha(RandomVariable):
#     def validate(self):
#         pass

#     def sample(self, n: int, **kwargs):
#         alpha_rv = TransformedVariable(
#             name="log_alpha_random_walk",
#             base_rv=RandomWalk(
#                 name="log_alpha",
#                 step_rv=DistributionalVariable(
#                     name="alpha_rw_step_rv", distribution=dist.Normal(0, 0.2)
#                 ),
#             ),
#             transforms=t.ExpTransform(),
#         )
#         alpha_init_rv = DistributionalVariable(
#             name="init_log_alpha", distribution=dist.Normal(jnp.log(3e10), 2)
#         )
#         init_alpha = alpha_init_rv.sample()
#         return alpha_rv.sample(n=n + n_initialization_points, init_vals=init_alpha, **kwargs)

# # alpha = TimeVaryingAlpha()
# alpha = DeterministicVariable('alpha', 3e10)
# # alpha = DistributionalVariable(
# #     name='alpha_init', distribution=dist.Normal(jnp.log(3e10), 0.05)
# # )

# #Latent wastewater
# latent_wastewater = latent.Wastewater(
#     infection_to_shedding_rv=inf_shed_int,
#     alpha_rv = alpha
# )

# sigma_rv = TransformedVariable(
#     "log_sigma_ww",
#     DistributionalVariable(
#         "log_sigma_ww_raw",
#         dist.Normal(loc=0.0, scale=1.0),
#     ),
#     transforms=t.ExpTransform(),  # ensures positivity
# )

# obs_ww = observation.LogNormalObservation(
#     "lognorm_rv",
#     sigma_rv=sigma_rv
# )

### Single Period Fitting functions

In [7]:
def single_fit(d: pd.DataFrame, latent_inf: RandomVariable,
               latent_severeprimary: RandomVariable, I0: RandomVariable,
               gen_int_rv: RandomVariable, rtproc: RandomVariable,
               obs_h: RandomVariable, obs_p: RandomVariable,
               num_warmup: int = 1000, num_samples: int = 1000,
               return_idata: bool = True, rng_seed: int = 54, num_chains: int = 2):
    """
    Fit the joint hospitalisation + primary care model to one window of data.

    return_idata=True  -> arviz InferenceData with posterior + posterior predictive (plots, diagnostics)
    return_idata=False -> dict of only the posterior sites estimate_hseek_infections / chains_agree need,
                          grouped by chain (arrays of shape (num_chains, num_samples, ...)); skips
                          posterior_predictive and the arviz conversion; use inside long loops
    rng_seed seeds the sampler (54 reproduces the earlier runs); fit_window_checked retries with other
    seeds when a chain gets stuck.
    """
    fit_model = model.SeverePrimaryModel(
        latent_infections_rv=latent_inf,
        latent_severe_primary_rv=latent_severeprimary,
        I0_rv=I0,
        gen_int_rv=gen_int_rv,
        Rt_process_rv=rtproc,
        severe_admission_obs_process_rv=obs_h,
        primary_care_cases_obs_process_rv=obs_p
    )

    # Run MCMC
    fit_model.run(
        num_samples=num_samples,
        num_warmup=num_warmup,
        rng_key=jax.random.key(rng_seed),
        mcmc_args=dict(progress_bar=False, num_chains=num_chains),
        data_observed_severe_admissions = d['new_hospitalisation_cases'].values,
        data_observed_primary_care_cases = d['est_covid_cases_adj'].values,
        weekly_obs = True
        )

    if not return_idata:
        keep = ('all_latent_infections', 'p_sym', 'p_hseek', 'I0')
        return {k: np.asarray(v) for k, v in fit_model.mcmc.get_samples(group_by_chain=True).items() if k in keep}

    ## Posterior predictive checks
    ppc_samples = fit_model.posterior_predictive(
        n_datapoints=len(d) * 7
    )

    idata = az.from_numpyro(
        posterior=fit_model.mcmc,
        posterior_predictive=ppc_samples
    )
    
    return(idata)

def estimate_hseek_infections(idata, n_inf_start, weekly = True):
    """
    Estimate the total number of infections that seek PRIMARY CARE.
    `idata` is an arviz InferenceData or the dict returned by single_fit(return_idata=False).
    Samples are flattened chain-major (sample s comes from chain s // num_samples).
    Returns an array of shape (n_weeks, n_samples) (or (n_days, n_samples) if weekly=False).
    """
    if isinstance(idata, dict):
        latent_inf_all = idata['all_latent_infections']
        latent_inf_all = latent_inf_all.reshape(-1, latent_inf_all.shape[-1])  # (chain*draw, n_inf)
        sym_samples = idata['p_sym'].reshape(-1)
        hseek_samples = idata['p_hseek'].reshape(-1)
    else:
        latent_inf_all = idata.posterior.all_latent_infections.values
        latent_inf_all = latent_inf_all.reshape(-1, latent_inf_all.shape[-1])  # (chain*draw, n_inf)
        sym_samples = idata.posterior.p_sym.values.reshape(-1)
        hseek_samples = idata.posterior.p_hseek.values.reshape(-1)

    n_samples, n_inf = latent_inf_all.shape
    n_weeks = int((n_inf - n_inf_start) / 7)

    scale_samples = sym_samples * hseek_samples

    latent_inf_samples = latent_inf_all[:, n_inf_start:]

    hseek_inf_samples = latent_inf_samples.T * scale_samples                  # (n_days, n_samples)

    if weekly:
    ##agg to weekly and sum
        hseek_inf_week_samples = hseek_inf_samples.reshape(n_weeks, 7 ,n_samples).sum(axis=1) ##It matters to shape into weeks first followed by days
    else:
        hseek_inf_week_samples = hseek_inf_samples.copy()
    return(hseek_inf_week_samples)


def get_agg_stats(post):
    """
    Median and 95% interval from posterior samples of shape (n_weeks, n_samples).
    """
    post = np.asarray(post)
    m = np.median(post, axis=1)
    q = np.quantile(post, q=[0.025, 0.975], axis=1)
    return(m, q)

## Rolling-window m3 estimator
`estimate_m3` fits a `window`-week model at every start position and stitches the first window's first `window-1` medians with the last-week median of every window (nothing is retained in memory and the posterior predictive step is skipped).

**Stuck-chain guard.** In some windows (e.g. those ending 2024-W12 to 2024-W17, just after the Dec-2023 wave) one chain settles in a degenerate mode: `I0` ≈ 2, random-walk sd ≈ 0 and the data explained by pure exponential growth, so the pooled last-week median comes out at 1e20+. `chains_agree` flags this (split R-hat of `I0`, and per-chain last-week medians differing by more than `ratio_max`) and `fit_window_checked` refits with the next seed in `SEEDS`; if every seed fails the estimate is NaN and a warning is printed. `refit_rows` re-estimates selected rows only (used by the loop below for partially filled columns).

In [8]:
from numpyro.diagnostics import split_gelman_rubin

SEEDS = (54, 2, 3, 5, 6, 7, 11, 13)   # tried in this order; 54 first so windows that were fine reproduce the earlier outputs


def chains_agree(samples, n_inf_start, rhat_max=1.1, ratio_max=2.0):
    """
    Stuck-chain check on a single_fit(return_idata=False) result.
    Returns (ok, rhat_I0, per-chain medians of the last-week m3).
    """
    i0 = np.asarray(samples['I0'])                                            # (chains, draws)
    rhat = float(split_gelman_rubin(i0))
    last = estimate_hseek_infections(samples, n_inf_start)[-1].reshape(i0.shape[0], -1)   # chain-major
    med = np.median(last, axis=1)
    ok = bool(np.isfinite(med).all() and rhat <= rhat_max and med.max() <= ratio_max * med.min())
    return ok, rhat, med


def fit_window_checked(d_sub, num_warmup=1000, num_samples=1000, seeds=SEEDS, label=''):
    """
    single_fit(return_idata=False) guarded by chains_agree: refitted with the next seed while a chain is stuck.
    Returns (samples, seed, n_tries); samples is None when every seed failed.
    """
    for k, seed in enumerate(seeds, start=1):
        samples = single_fit(d_sub, latent_inf, latent_severeprimary, I0, gen_int_rv, rtproc, obs_h, obs_p,
                             num_warmup=num_warmup, num_samples=num_samples, return_idata=False, rng_seed=seed)
        ok, rhat, med = chains_agree(samples, n_initialization_points)
        if ok:
            return samples, seed, k
        print(f"{label}: stuck chain with seed {seed} (R-hat(I0) = {rhat:.2f}, per-chain m3 = "
              f"{', '.join(f'{m:.3g}' for m in med)}); retrying", flush=True)
    print(f"{label}: every seed in {tuple(seeds)} left a stuck chain; estimate set to NaN", flush=True)
    return None, None, len(seeds)


def window_medians(d_main, end, window, num_warmup, num_samples, seeds, label):
    """Weekly m3 medians (length window) of the window ending at row `end`; None if every seed failed."""
    d_sub = d_main.iloc[end - window + 1:end + 1]
    samples, seed, tries = fit_window_checked(d_sub, num_warmup, num_samples, seeds, label)
    if samples is None:
        return None, seed, tries
    m, _ = get_agg_stats(estimate_hseek_infections(samples, n_inf_start=n_initialization_points))
    del samples
    return m, seed, tries


def _log_fit(label, d_main, end, value, seed, tries, t0):
    last = d_main.iloc[end]
    print(f"{label}: fitted up to {int(last['year'])}-W{int(last['epi_week']):02d}, m3 = {value:.0f} "
          f"(seed {seed}, {tries} {'try' if tries == 1 else 'tries'}, {time.time() - t0:.0f}s)", flush=True)


def estimate_m3(d_main, window=20, num_warmup=1000, num_samples=1000, verbose=True, label='', seeds=SEEDS):
    """
    Rolling-window m3 estimate for every week of d_main.
    Returns np.ndarray of length len(d_main): the first window's first (window-1) medians,
    followed by the last-week median of every window. Keeps no idata objects.
    Every window goes through fit_window_checked (NaN where every seed left a stuck chain).
    verbose=True prints a line after every window fit (label is prefixed, e.g. 'n150 sim1').
    """
    n_start = np.arange(0, len(d_main) - window + 1)
    first_med = np.full(window - 1, np.nan)
    last_med = np.full(len(n_start), np.nan)
    for i, s in enumerate(n_start):
        t0 = time.time()
        end = s + window - 1
        lab = f"{label} window {i + 1}/{len(n_start)}"
        m, seed, tries = window_medians(d_main, end, window, num_warmup, num_samples, seeds, lab)
        if m is not None:
            if i == 0:
                first_med = m[:-1]
            last_med[i] = m[-1]
        if verbose:
            _log_fit(lab, d_main, end, last_med[i], seed, tries, t0)
    m3 = np.concatenate([first_med, last_med])
    assert m3.shape == (len(d_main),)
    return m3


def refit_rows(d_main, rows, window=20, num_warmup=1000, num_samples=1000, verbose=True, label='', seeds=SEEDS):
    """
    Re-estimate m3 for the given row positions of d_main only, with the same construction as estimate_m3
    (rows below window-1 all come from the first window, which is fitted once).
    Returns {row: value}; value is NaN where every seed left a stuck chain.
    """
    rows = sorted({int(r) for r in rows})
    ends = sorted({max(r, window - 1) for r in rows})
    fitted = {}
    for j, end in enumerate(ends):
        t0 = time.time()
        lab = f"{label} refit {j + 1}/{len(ends)}"
        m, seed, tries = window_medians(d_main, end, window, num_warmup, num_samples, seeds, lab)
        fitted[end] = m
        if verbose:
            _log_fit(lab, d_main, end, np.nan if m is None else m[-1], seed, tries, t0)
    out = {}
    for r in rows:
        m = fitted[max(r, window - 1)]
        out[r] = np.nan if m is None else float(m[-1] if r >= window - 1 else m[r])
    return out

In [9]:
# Regenerate true_m3.csv with the same machinery (single positivity series).
# Left commented so "Run All" does not spend hours here; uncomment to run.
# smoothed = pd.read_csv('../sg_data/sample_size/smoothed_covid_pos.csv')['Smoothed_COVID19']   # 157 rows, 2023.0 - 2026.0
# d_true = apply_positivity(covid_ari_hosp_d, smoothed)      # or covid_ari_hosp_d.iloc[:N_WEEKS] for observed positivity
# all_val = estimate_m3(d_true, window=20, verbose=True)
# pd.DataFrame({'year': d_true['year'], 'epi_week': d_true['epi_week'], 'm3_estimate': all_val}).to_csv("../sample_size_outputs/true_m3.csv")


## Loop over sample-size files
For each `sim_data_n*.csv`, run `estimate_m3` on every `sim*` column and write `m3_est_n*.csv` (`Year, Week, sim1..sim100`, mirroring the input). The file is rewritten after **every column**. On re-run a column is skipped when complete, fitted in full when empty, and only its NaN cells are re-fitted (`refit_rows`) when partially filled, so the loop can be stopped and resumed at any time and specific cells can be redone by setting them to NaN in the CSV. Narrow `SAMPLE_FILES` / `SIM_COLUMNS` for a subset. To parallelise, run one copy of the notebook per file (never two kernels on the same file).

In [11]:
import re, gc, time
from pathlib import Path

# ---- parameters ----
SAMPLE_FILES = sorted(Path('../sg_data/sample_size').glob('sim_data_n*.csv'))   # narrow to a subset here
SIM_COLUMNS  = None            # None = all sim* columns; or e.g. ['sim1', 'sim2']
WINDOW       = 20
NUM_WARMUP, NUM_SAMPLES = 1000, 1000
OUT_DIR      = Path('../sample_size_outputs')
VERBOSE      = True            # True prints a line after every window fit; False only per column
# ---------------------

def size_stub(path):
    """'sim_data_n150.csv' -> 'n150'"""
    return re.search(r'_(n\d+)\.csv$', path.name).group(1)

def sim_columns(df):
    """sim* columns in FILE order (never sorted: that would give sim1, sim10, sim100, ...)."""
    return [c for c in df.columns if re.fullmatch(r'sim\d+', c)]

def load_or_init(out_path, sim_df, sim_cols):
    """Existing output (to resume) or a fresh Year/Week frame; all sim columns present, NaN until filled."""
    out = pd.read_csv(out_path) if out_path.exists() else sim_df[['Year', 'Week']].copy()
    assert (out['Year'].values == sim_df['Year'].values).all() and (out['Week'].values == sim_df['Week'].values).all(), \
        f'{out_path.name} does not line up with {len(sim_df)} Year/Week rows of the sample file'
    missing = [c for c in sim_cols if c not in out.columns]
    if missing:                                     # add all placeholders at once (avoids a fragmented frame)
        out = pd.concat([out, pd.DataFrame(np.nan, index=out.index, columns=missing)], axis=1)
    return out[['Year', 'Week'] + sim_cols]        # fixed column order regardless of completion order

def write_atomic(df, path):
    tmp = path.with_suffix(path.suffix + '.tmp')
    df.to_csv(tmp, index=False)
    tmp.replace(path)

OUT_DIR.mkdir(parents=True, exist_ok=True)
for f in SAMPLE_FILES:
    stub = size_stub(f)
    sim_df = pd.read_csv(f).iloc[:N_WEEKS].reset_index(drop=True)
    assert len(sim_df) == N_WEEKS, f'{f.name}: fewer than {N_WEEKS} rows'
    assert (sim_df['Year'].values == covid_ari_hosp_d['year'].values[:N_WEEKS]).all()
    assert (sim_df['Week'].values == covid_ari_hosp_d['epi_week'].values[:N_WEEKS]).all()
    cols = sim_columns(sim_df)
    todo = cols if SIM_COLUMNS is None else [c for c in cols if c in SIM_COLUMNS]

    out_path = OUT_DIR / f'm3_est_{stub}.csv'
    out = load_or_init(out_path, sim_df, cols)
    n_done = int(out[todo].notna().all().sum())
    n_partial = int((out[todo].notna().any() & out[todo].isna().any()).sum())
    print(f'{stub}: {len(todo)} columns to run, {n_done} already done, {n_partial} partial -> {out_path}')

    for col in todo:
        nan_rows = np.flatnonzero(out[col].isna().values)
        if len(nan_rows) == 0:
            print(f'{stub} {col}: already done, skipping')
            continue
        t0 = time.time()
        d_col = apply_positivity(covid_ari_hosp_d, sim_df[col].values)
        if len(nan_rows) == len(out):                # empty column: full rolling-window run
            out[col] = estimate_m3(d_col, window=WINDOW, num_warmup=NUM_WARMUP, num_samples=NUM_SAMPLES,
                                   verbose=VERBOSE, label=f'{stub} {col}', seeds=SEEDS)
        else:                                        # partial column: re-fit only the NaN cells
            vals = refit_rows(d_col, nan_rows, window=WINDOW, num_warmup=NUM_WARMUP, num_samples=NUM_SAMPLES,
                              verbose=VERBOSE, label=f'{stub} {col}', seeds=SEEDS)
            out.loc[list(vals), col] = list(vals.values())
        write_atomic(out, out_path)                  # partial output saved after every column
        gc.collect(); jax.clear_caches()
        print(f'{stub} {col}: {time.time() - t0:.0f}s, {int(out[col].isna().sum())} NaN left', flush=True)

n150: 100 columns to run, 100 already done, 0 partial -> ../sample_size_outputs/m3_est_n150.csv
n150 sim1: already done, skipping
n150 sim2: already done, skipping
n150 sim3: already done, skipping
n150 sim4: already done, skipping
n150 sim5: already done, skipping
n150 sim6: already done, skipping
n150 sim7: already done, skipping
n150 sim8: already done, skipping
n150 sim9: already done, skipping
n150 sim10: already done, skipping
n150 sim11: already done, skipping
n150 sim12: already done, skipping
n150 sim13: already done, skipping
n150 sim14: already done, skipping
n150 sim15: already done, skipping
n150 sim16: already done, skipping
n150 sim17: already done, skipping
n150 sim18: already done, skipping
n150 sim19: already done, skipping
n150 sim20: already done, skipping
n150 sim21: already done, skipping
n150 sim22: already done, skipping
n150 sim23: already done, skipping
n150 sim24: already done, skipping
n150 sim25: already done, skipping
n150 sim26: already done, skipping
n15

KeyboardInterrupt: 

### Exploratory cells (below)
Kept as they were; they are not part of the sample-size loop and are currently not runnable as-is: the next cell needs `all_val` from the single-series cell above, the wave-period fit references an undefined `covid_hosp_d` (should be `covid_ari_hosp_d`), and the last cells need `d_post`, `m`, `q` from a manual `single_fit(...)` (e.g. `d_post = single_fit(covid_ari_hosp_d.iloc[:20], latent_inf, latent_severeprimary, I0, gen_int_rv, rtproc, obs_h, obs_p)`).

In [ ]:
np.where(all_val > 1e5)